In [0]:
# Set working environment
catalog_name = "f1_dev" 
schemas = ["bronze", "silver", "gold"]

# Create the catalog if it does not exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

# Create each schema inside the catalog
for schema_name in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

In [0]:
import os
import pandas as pd

catalog_name = "f1_dev"
data_dir = "data/"

csv_files = {
    "circuits": "circuits.csv",
    "constructor_standings": "constructor_standings.csv", 
    "constructors": "constructors.csv",
    "driver_standings": "driver_standings.csv",
    "drivers": "drivers.csv",
    "lap_times": "lap_times.csv",
    "pit_stops": "pit_stops.csv",
    "qualifying": "qualifying.csv",
    "races": "races.csv",
    "results": "results.csv",
    "seasons": "seasons.csv",
    "sprint_results": "sprint_results.csv",
    "status": "status.csv",
}

for table_name, file_name in csv_files.items():
    file_path = os.path.join(data_dir, file_name)
    spark_df = spark.createDataFrame(pd.read_csv(file_path))
    spark_df.write.mode("overwrite").saveAsTable(f"{catalog_name}.bronze.{table_name}")
    print(f"Created table: {catalog_name}.bronze.{table_name}")

print("Bronze ingestion complete!")

### **Apply Data Classifications and Tags**

First, we'll tag our tables with sensitivity levels. Table properties are metadata tags that help classify data sensitivity without changing the actual data structure:

- Metadata key-value pairs attached to tables
- Used by governance tools to understand data sensitivity
- Don't affect table structure or data
- Can be queried programmatically for compliance

Our data classification schema:
 - Public: Anyone can access 
 - Internal: Employees only 
 - Confidential: Limited access 
 - Restricted: Highly sensitive 

Adding classification tags like "Confidential," "Restricted," or "Public" to Unity Catalog serve as metadata that make your data assets more manageable, secure, and compliant with both internal policies and external regulations.


In [0]:
# Define classification tags for bronze tables
# Format: (table_name, classification, contains_pii)
tables_config = [
    ("f1_dev.bronze.circuits", "public", "false"),
    ("f1_dev.bronze.constructor_standings", "public", "false"),
    ("f1_dev.bronze.constructors", "public", "false"),
    ("f1_dev.bronze.driver_standings", "public", "false"),
    ("f1_dev.bronze.drivers", "public", "true"),  # Contains driver names & DOB
    ("f1_dev.bronze.lap_times", "public", "false"),
    ("f1_dev.bronze.pit_stops", "public", "false"),
    ("f1_dev.bronze.qualifying", "public", "false"),
    ("f1_dev.bronze.races", "public", "false"),
    ("f1_dev.bronze.results", "public", "false"),
    ("f1_dev.bronze.seasons", "public", "false"),
    ("f1_dev.bronze.sprint_results", "public", "false"),
    ("f1_dev.bronze.status", "public", "false")
]

# Apply tags in Unity Catalog
for table_name, classification, has_pii in tables_config:
    spark.sql(f"""
        ALTER TABLE {table_name} SET TAGS (
            'classification' = '{classification}',
            'contains_pii' = '{has_pii}'
        )
    """)
    print(f"✓ Tagged {table_name} (Classification: {classification}, PII: {has_pii})")